# Adaptive Road Vision — JSONL Annotation Processing

This notebook converts the BDD100K JSON/JSONL annotations into a clean, project-specific metadata table for the next stages of Adaptive Road Vision.

### Main goals
- Read BDD100K JSON/JSONL annotations without changing the original dataset.
- Extract image/frame identifiers and object annotations when available.
- Extract Day/Night and weather/condition attributes when available.
- Normalize condition names to the project vocabulary: **Day, Night, Rain, Fog, Clear/Other**.
- Produce a processed metadata file for later condition analysis and CycleGAN source selection.

**Important:** This notebook does not create synthetic fog data and does not modify the original BDD100K annotations.

## 1. Project Configuration

In [1]:
from pathlib import Path
import json
import re
import csv
from collections import Counter, defaultdict

PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
BDD_ROOT = PROJECT_ROOT / "datasets" / "raw" / "BDD100K"
LABEL_ROOT = BDD_ROOT / "labels"
IMAGE_ROOT = BDD_ROOT / "images" / "100k"
PROCESSED_ROOT = PROJECT_ROOT / "datasets" / "processed"
METADATA_ROOT = PROCESSED_ROOT / "metadata"

METADATA_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "val", "test"]
print("Project:", PROJECT_ROOT)
print("BDD100K:", BDD_ROOT)
print("Labels :", LABEL_ROOT)
print("Output :", METADATA_ROOT)

Project: C:\Users\ankit\Desktop\Adaptive_Road_Vision
BDD100K: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K
Labels : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels
Output : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata


## 2. Find JSON / JSONL Annotation Files

In [2]:
ANNOTATION_EXTENSIONS = {".json", ".jsonl"}

def find_annotation_files(folder):
    if not folder.exists():
        return []
    return sorted([
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in ANNOTATION_EXTENSIONS
    ])

annotation_files = {split: find_annotation_files(LABEL_ROOT / split) for split in SPLITS}

for split in SPLITS:
    print(f"{split.upper():5}: {len(annotation_files[split]):,} files")
    for p in annotation_files[split][:10]:
        print("  ", p.name)

all_annotation_files = [p for split in SPLITS for p in annotation_files[split]]
print("\nTotal annotation files:", len(all_annotation_files))

TRAIN: 70,000 files
   0000f77c-6257be58.json
   0000f77c-62c2a288.json
   0000f77c-cb820c98.json
   0001542f-5ce3cf52.json
   0001542f-7c670be8.json
   0001542f-ec815219.json
   0004974f-05e1c285.json
   00054602-3bf57337.json
   00067cfb-5443fe39.json
   00067cfb-5adfaaa7.json
VAL  : 10,000 files
   b1c66a42-6f7d68ca.json
   b1c81faa-3df17267.json
   b1c81faa-c80764c5.json
   b1c9c847-3bda4659.json
   b1ca2e5d-84cf9134.json
   b1ca8418-84a133a0.json
   b1cac6a7-04e33135.json
   b1cd1e94-26dd524f.json
   b1cd1e94-549d0bfe.json
   b1ceb32e-3f481b43.json
TEST : 20,000 files
   cabc30fc-e7726578.json
   cabc30fc-eb673c5a.json
   cabc30fc-fd79926f.json
   cabc9045-1b8282ba.json
   cabc9045-581f64de.json
   cabc9045-5a50690f.json
   cabc9045-b3349548.json
   cabc9045-c6dc9529.json
   cabc9045-cd422b81.json
   cabc9045-d91ecb66.json

Total annotation files: 100000


## 3. Flexible JSON / JSONL Reader

In [3]:
def iter_json_records(path):
    """Yield records from JSON or JSONL without assuming one fixed BDD100K layout."""
    with path.open("r", encoding="utf-8-sig") as f:
        text_start = f.read(2048)
        f.seek(0)
        stripped = text_start.lstrip()

        if not stripped:
            return

        # Standard JSON: object or array
        if path.suffix.lower() == ".json" and stripped[:1] in "[{":
            try:
                data = json.load(f)
                if isinstance(data, list):
                    for item in data:
                        yield item
                elif isinstance(data, dict):
                    yield data
                return
            except json.JSONDecodeError:
                f.seek(0)

        # JSONL or JSON with one object per line
        for line_number, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue

print("Reader ready.")

Reader ready.


## 4. Inspect the First Real Annotation Record

In [4]:
first_record = None
first_file = None

for split in SPLITS:
    if annotation_files[split]:
        first_file = annotation_files[split][0]
        try:
            first_record = next(iter_json_records(first_file), None)
        except Exception as e:
            print("Read error:", e)
        break

print("First annotation file:", first_file)
if isinstance(first_record, dict):
    print("Top-level keys:", list(first_record.keys()))
    print("\nSample structure:")
    print(json.dumps(first_record, indent=2)[:6000])
else:
    print("No dictionary record was found.")

First annotation file: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\train\0000f77c-6257be58.json
Top-level keys: ['name', 'frames', 'attributes']

Sample structure:
{
  "name": "0000f77c-6257be58",
  "frames": [
    {
      "timestamp": 10000,
      "objects": [
        {
          "category": "traffic light",
          "id": 0,
          "attributes": {
            "occluded": false,
            "truncated": false,
            "trafficLightColor": "green"
          },
          "box2d": {
            "x1": 1125.902264,
            "y1": 133.184488,
            "x2": 1156.978645,
            "y2": 210.875445
          }
        },
        {
          "category": "traffic light",
          "id": 1,
          "attributes": {
            "occluded": false,
            "truncated": false,
            "trafficLightColor": "green"
          },
          "box2d": {
            "x1": 1156.978645,
            "y1": 136.637417,
            "x2": 1191.50796,
           

## 5. Recursive Attribute Extraction

In [5]:
def normalize_key(value):
    return str(value).lower().strip().replace("-", "_").replace(" ", "_")

def recursive_find_values(obj, target_keys):
    """Find values for target keys anywhere inside nested dictionaries/lists."""
    found = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            key_norm = normalize_key(key)
            if key_norm in target_keys:
                found.append(value)
            found.extend(recursive_find_values(value, target_keys))
    elif isinstance(obj, list):
        for value in obj:
            found.extend(recursive_find_values(value, target_keys))
    return found

WEATHER_KEYS = {
    "weather", "weather_condition", "scene_weather", "weathercondition"
}
TIME_KEYS = {
    "timeofday", "time_of_day", "time", "lighting", "scene_time"
}
IMAGE_KEYS = {
    "name", "filename", "file_name", "image", "image_name", "image_path", "frame_id"
}

print("Attribute extraction helpers ready.")

Attribute extraction helpers ready.


## 6. Normalize Day/Night and Weather Conditions

In [6]:
def first_text(values):
    for value in values:
        if isinstance(value, str) and value.strip():
            return value.strip()
        if isinstance(value, (int, float)):
            return str(value)
    return "Unknown"

def normalize_time(value):
    s = str(value).lower().strip()
    if "night" in s or "dawn" in s or "dusk" in s:
        return "Night"
    if "day" in s:
        return "Day"
    return "Unknown"

def normalize_weather(value):
    s = str(value).lower().strip()
    if "fog" in s or "haze" in s:
        return "Fog"
    if "rain" in s or "drizzle" in s or "storm" in s:
        return "Rain"
    if "snow" in s:
        return "Snow"
    if "clear" in s or "sunny" in s or "cloudy" in s or "overcast" in s:
        return "Clear"
    return "Unknown"

print("Condition normalization ready.")

Condition normalization ready.


## 7. Extract Object Labels

In [7]:
OBJECT_CONTAINER_KEYS = {"labels", "objects", "annotations", "detections", "frames"}
OBJECT_CLASS_KEYS = {"category", "category_name", "class", "class_name", "name", "label", "type"}

def extract_object_classes(obj):
    classes = []

    def walk(value):
        if isinstance(value, dict):
            # Look for a class/category-like field.
            for key, val in value.items():
                if normalize_key(key) in OBJECT_CLASS_KEYS and isinstance(val, str):
                    classes.append(val.strip())
            for val in value.values():
                walk(val)
        elif isinstance(value, list):
            for item in value:
                walk(item)

    walk(obj)
    return classes

print("Object-label extraction ready.")

Object-label extraction ready.


## 8. Build One Metadata Record from an Annotation

In [8]:
def extract_image_name(record, source_file):
    values = recursive_find_values(record, IMAGE_KEYS)
    for value in values:
        if isinstance(value, str) and value.strip():
            return Path(value.strip()).name
    return source_file.stem

def annotation_to_record(record, split, source_file):
    weather_values = recursive_find_values(record, WEATHER_KEYS)
    time_values = recursive_find_values(record, TIME_KEYS)

    weather_raw = first_text(weather_values)
    time_raw = first_text(time_values)

    objects = extract_object_classes(record)

    image_name = extract_image_name(record, source_file)
    return {
        "split": split,
        "image_name": image_name,
        "weather_raw": weather_raw,
        "weather": normalize_weather(weather_raw),
        "time_raw": time_raw,
        "time_of_day": normalize_time(time_raw),
        "object_count": len(objects),
        "object_classes": "|".join(sorted(set(objects))),
        "annotation_file": str(source_file.relative_to(BDD_ROOT)),
    }

print("Metadata record builder ready.")

Metadata record builder ready.


## 9. Process Annotation Files

In [9]:
records = []
parse_errors = []
empty_files = []

for split in SPLITS:
    print(f"Processing {split}...")
    for path in annotation_files[split]:
        try:
            file_records = list(iter_json_records(path))
            if not file_records:
                empty_files.append(path)
                continue
            for record in file_records:
                if isinstance(record, dict):
                    records.append(annotation_to_record(record, split, path))
        except Exception as e:
            parse_errors.append((path, repr(e)))

print("\nRecords extracted:", f"{len(records):,}")
print("Empty annotation files:", len(empty_files))
print("Parse errors:", len(parse_errors))

Processing train...
Processing val...
Processing test...

Records extracted: 100,000
Empty annotation files: 0
Parse errors: 0


## 10. Inspect Processed Records

In [10]:
for record in records[:10]:
    print(record)

{'split': 'train', 'image_name': '0000f77c-6257be58', 'weather_raw': 'clear', 'weather': 'Clear', 'time_raw': 'daytime', 'time_of_day': 'Day', 'object_count': 12, 'object_classes': '0000f77c-6257be58|area/drivable|car|lane/road curb|traffic light|traffic sign', 'annotation_file': 'labels\\train\\0000f77c-6257be58.json'}
{'split': 'train', 'image_name': '0000f77c-62c2a288', 'weather_raw': 'clear', 'weather': 'Clear', 'time_raw': 'dawn/dusk', 'time_of_day': 'Night', 'object_count': 18, 'object_classes': '0000f77c-62c2a288|area/alternative|area/drivable|lane/road curb|lane/single white|person|traffic sign', 'annotation_file': 'labels\\train\\0000f77c-62c2a288.json'}
{'split': 'train', 'image_name': '0000f77c-cb820c98', 'weather_raw': 'clear', 'weather': 'Clear', 'time_raw': 'dawn/dusk', 'time_of_day': 'Night', 'object_count': 13, 'object_classes': '0000f77c-cb820c98|area/drivable|car|lane/double yellow|lane/road curb', 'annotation_file': 'labels\\train\\0000f77c-cb820c98.json'}
{'split': 

## 11. Condition Distribution

In [11]:
time_counter = Counter(r["time_of_day"] for r in records)
weather_counter = Counter(r["weather"] for r in records)
condition_counter = Counter((r["weather"], r["time_of_day"]) for r in records)

print("TIME OF DAY")
for key, value in time_counter.most_common():
    print(f"{key:10}: {value:,}")

print("\nWEATHER")
for key, value in weather_counter.most_common():
    print(f"{key:10}: {value:,}")

print("\nWEATHER × TIME")
for (weather, time_of_day), value in sorted(condition_counter.items(), key=lambda x: (-x[1], x[0])):
    print(f"{weather:10} + {time_of_day:8}: {value:,}")

TIME OF DAY
Day       : 52,504
Night     : 47,280
Unknown   : 216

WEATHER
Clear     : 73,170
Unknown   : 11,631
Snow      : 7,891
Rain      : 7,127
Fog       : 181

WEATHER × TIME
Clear      + Night   : 38,417
Clear      + Day     : 34,749
Unknown    + Day     : 10,060
Snow       + Day     : 4,085
Snow       + Night   : 3,803
Rain       + Night   : 3,570
Rain       + Day     : 3,542
Unknown    + Night   : 1,377
Unknown    + Unknown : 194
Fog        + Night   : 113
Fog        + Day     : 68
Rain       + Unknown : 15
Clear      + Unknown : 4
Snow       + Unknown : 3


## 12. Project Condition Groups

In [12]:
def project_condition(row):
    weather = row["weather"]
    tod = row["time_of_day"]

    if weather == "Fog":
        return "Fog"
    if weather == "Rain":
        return "Rain"
    if tod == "Night":
        return "Night"
    if tod == "Day":
        return "Day"
    return "Unknown"

for row in records:
    row["project_condition"] = project_condition(row)

project_condition_counter = Counter(r["project_condition"] for r in records)
print("Project condition distribution:")
for condition, count in project_condition_counter.most_common():
    print(f"{condition:10}: {count:,}")

Project condition distribution:
Day       : 48,894
Night     : 43,597
Rain      : 7,127
Unknown   : 201
Fog       : 181


## 13. Detect Matching Image Files

This creates an index of image filenames and checks how many processed annotation records can be linked to an actual BDD100K image.

In [13]:
image_index = {}
for split in SPLITS:
    folder = IMAGE_ROOT / split
    if folder.exists():
        for p in folder.rglob('*'):
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                image_index.setdefault((split, p.name), p)

matched = 0
unmatched = []
for row in records:
    key = (row["split"], row["image_name"])
    if key in image_index:
        row["image_path"] = str(image_index[key])
        matched += 1
    else:
        row["image_path"] = ""
        unmatched.append(row)

print(f"Matched records  : {matched:,}")
print(f"Unmatched records: {len(unmatched):,}")
for row in unmatched[:20]:
    print(row["split"], row["image_name"], "<-", row["annotation_file"])

Matched records  : 0
Unmatched records: 100,000
train 0000f77c-6257be58 <- labels\train\0000f77c-6257be58.json
train 0000f77c-62c2a288 <- labels\train\0000f77c-62c2a288.json
train 0000f77c-cb820c98 <- labels\train\0000f77c-cb820c98.json
train 0001542f-5ce3cf52 <- labels\train\0001542f-5ce3cf52.json
train 0001542f-7c670be8 <- labels\train\0001542f-7c670be8.json
train 0001542f-ec815219 <- labels\train\0001542f-ec815219.json
train 0004974f-05e1c285 <- labels\train\0004974f-05e1c285.json
train 00054602-3bf57337 <- labels\train\00054602-3bf57337.json
train 00067cfb-5443fe39 <- labels\train\00067cfb-5443fe39.json
train 00067cfb-5adfaaa7 <- labels\train\00067cfb-5adfaaa7.json
train 00067cfb-caba8a02 <- labels\train\00067cfb-caba8a02.json
train 00067cfb-e535423e <- labels\train\00067cfb-e535423e.json
train 00067cfb-f1b91e3c <- labels\train\00067cfb-f1b91e3c.json
train 0008a165-c48f4b3e <- labels\train\0008a165-c48f4b3e.json
train 00091078-59817bb0 <- labels\train\00091078-59817bb0.json
train 0

In [14]:
## 13. Detect Matching Image Files — FIXED

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def normalize_image_id(value):
    """
    Convert an image/annotation identifier into a comparable ID.
    Example:
        0000f77c-6257be58.jpg
        0000f77c-6257be58
    both become:
        0000f77c-6257be58
    """
    if value is None:
        return ""

    value = str(value).strip()

    # Convert Windows/Linux path to filename
    value = Path(value).name

    # Remove image extension
    value = Path(value).stem

    return value.lower()


# ---------------------------------------------------------
# Build image index
# ---------------------------------------------------------

image_index = {}

for split in SPLITS:

    folder = IMAGE_ROOT / split

    if not folder.exists():
        print(f"WARNING: {folder} does not exist")
        continue

    for p in folder.rglob("*"):

        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:

            image_id = normalize_image_id(p.name)

            image_index[(split, image_id)] = p


print("Total indexed images:", len(image_index))


# ---------------------------------------------------------
# Match annotation records with images
# ---------------------------------------------------------

matched = 0
unmatched = []

for row in records:

    annotation_id = normalize_image_id(row["image_name"])

    key = (row["split"], annotation_id)

    if key in image_index:

        row["image_path"] = str(image_index[key])

        # Store normalized ID for future processing
        row["image_id"] = annotation_id

        matched += 1

    else:

        row["image_path"] = ""
        row["image_id"] = annotation_id

        unmatched.append(row)


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("=" * 70)
print("IMAGE ↔ ANNOTATION MATCHING")
print("=" * 70)

print(f"Indexed images     : {len(image_index):,}")
print(f"Annotation records : {len(records):,}")
print(f"Matched records    : {matched:,}")
print(f"Unmatched records  : {len(unmatched):,}")


# Show first unmatched records if any

if unmatched:

    print("\nFirst unmatched records:")

    for row in unmatched[:20]:

        print(
            f"{row['split']:5} | "
            f"{row['image_name']} | "
            f"{row['annotation_file']}"
        )

Total indexed images: 100000
IMAGE ↔ ANNOTATION MATCHING
Indexed images     : 100,000
Annotation records : 100,000
Matched records    : 100,000
Unmatched records  : 0


## 14. Split-wise Metadata Summary

In [15]:
split_summary = defaultdict(Counter)
for row in records:
    split_summary[row["split"]][row["project_condition"]] += 1

for split in SPLITS:
    print(f"\n{split.upper()}")
    for condition, count in split_summary[split].most_common():
        print(f"  {condition:10}: {count:,}")


TRAIN
  Day       : 34,220
  Night     : 30,441
  Rain      : 5,083
  Fog       : 130
  Unknown   : 126

VAL
  Day       : 4,857
  Night     : 4,358
  Rain      : 738
  Unknown   : 34
  Fog       : 13

TEST
  Day       : 9,817
  Night     : 8,798
  Rain      : 1,306
  Unknown   : 41
  Fog       : 38


## 15. Save Processed Metadata

The processed metadata is an output of this stage. It is safe to regenerate later and does not modify the raw BDD100K files.

In [16]:
import pandas as pd

if records:
    metadata_df = pd.DataFrame(records)
    metadata_csv = METADATA_ROOT / "bdd100k_processed_metadata.csv"
    metadata_json = METADATA_ROOT / "bdd100k_processed_metadata.json"

    metadata_df.to_csv(metadata_csv, index=False, encoding="utf-8")
    metadata_df.to_json(metadata_json, orient="records", indent=2, force_ascii=False)

    print("Saved:")
    print(" ", metadata_csv)
    print(" ", metadata_json)
    print("Rows:", len(metadata_df))
    print("Columns:", list(metadata_df.columns))
else:
    print("No records were extracted, so no metadata output was created.")

Saved:
  C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\bdd100k_processed_metadata.csv
  C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\bdd100k_processed_metadata.json
Rows: 100000
Columns: ['split', 'image_name', 'weather_raw', 'weather', 'time_raw', 'time_of_day', 'object_count', 'object_classes', 'annotation_file', 'project_condition', 'image_path', 'image_id']


## 16. Processing Quality Report

This report summarizes what the parser could establish. Unknown condition values are preserved rather than guessed.

In [17]:
print("=" * 70)
print("JSON/JSONL PROCESSING REPORT")
print("=" * 70)
print(f"Annotation files found : {len(all_annotation_files):,}")
print(f"Records extracted      : {len(records):,}")
print(f"Empty files            : {len(empty_files):,}")
print(f"Parse errors           : {len(parse_errors):,}")
print(f"Image matches          : {matched:,}")
print(f"Image mismatches       : {len(unmatched):,}")
print("\nCondition counts:")
for condition, count in project_condition_counter.most_common():
    print(f"  {condition:10}: {count:,}")

if parse_errors:
    print("\nFirst parse errors:")
    for path, error in parse_errors[:10]:
        print(" ", path, "->", error)

JSON/JSONL PROCESSING REPORT
Annotation files found : 100,000
Records extracted      : 100,000
Empty files            : 0
Parse errors           : 0
Image matches          : 100,000
Image mismatches       : 0

Condition counts:
  Day       : 48,894
  Night     : 43,597
  Rain      : 7,127
  Unknown   : 201
  Fog       : 181


## 17. Important Interpretation Rules

- `Unknown` is kept when the annotation source does not explicitly provide a usable condition.
- The notebook does **not** invent Fog labels from image appearance.
- The notebook does **not** create synthetic fog images.
- The original 70K/10K/20K split remains unchanged.
- CycleGAN source selection should use **training data only** after this metadata has been checked.
- Before training the condition model, inspect the generated metadata and confirm that the extracted condition labels match the actual BDD100K annotation structure.

## Next Notebook

After checking this notebook's output, continue with:

**`04_condition_label_preparation.ipynb`**

That stage will prepare the clean Day/Night/Rain/Fog condition-label dataset needed by the condition-analysis module and the CycleGAN source-selection workflow.